# Workflow: Xenium + Visium {#sec-crs-workflow-xenvis}



## Preamble

### Dependencies

In [ ]:
library(BayesSpace)
library(ggspavis)
library(harmony)
library(RANN)
library(scater)
library(scrapper)
library(OSTA.data)
library(patchwork)
library(SpatialExperiment)
library(SpatialExperimentIO)
library(VisiumIO)
# set seed for random number generation
# in order to make results reproducible
set.seed(194849)

### Introduction

In this demo, we will rely on data from @Janesick2023-high-res, which includes
same-section Visium and Xenium measurements on human breast cancer tissue.

![H&E staining](../images/xenvis-visium.png){height=300px}
![Xenium IF](../images/xenvis-xenium.png){height=300px}

## Setup {#sec-crs-workflow-xenvis-load-data}

We start out by retrieving these datasets from the OSF repository, and reading 
them into R as separate `r BiocStyle::Biocpkg("SpatialExperiment")` objects:

In [ ]:
# Visium
id <- "Visium_HumanBreast_Janesick"
pa <- OSTA.data_load(id)
dir.create(td <- tempfile())
unzip(pa, exdir=td)
obj <- TENxVisium(
    spacerangerOut=file.path(td, "outs"), 
    images="lowres", 
    format="h5")
(vis <- import(obj))

# Xenium
id <- "Xenium_HumanBreast1_Janesick"
pa <- OSTA.data_load(id, mol=FALSE)
dir.create(td <- tempfile())
unzip(pa, exdir=td)
(xen <- readXeniumSXE(td, addTx=FALSE))

# also retrieve cell subpopulation labels
df <- read.csv(file.path(td, "annotation.csv"))
xen$anno <- df$Annotation[match(xen$cell_id, df$Barcode)]

We'll also do some data wrangling to simplify spatial coordinate names, and use
gene symbols (rather than ensembl identifiers) as feature names for both data:

In [ ]:
# simplify spatial coordinate names
spatialCoordsNames(vis) <- spatialCoordsNames(xen) <- c("x", "y")
# use gene symbols as feature names
rownames(vis) <- make.unique(rowData(vis)$Symbol)
rownames(xen) <- make.unique(rowData(xen)$Symbol)

## Alignment

To align Xenium and Visium sections, we use the affine transformation matrix 
provided by 10x Genomics, which was obtained by registration of Xenium onto 
Visium in Python with the [Fiji Java plug-in](https://www.10xgenomics.com/resources/analysis-guides/he-to-xenium-dapi-image-registration-with-fiji); see
@sec-crs-spatial-registration for details.

In [ ]:
# affine matrix for aligning Xenium onto Visium
mtx <- matrix(nrow=2, byrow=TRUE, c(
    8.82797498e-02, -1.91831377e+00, 1.63476055e+04,
    1.84141210e+00,  5.96797885e-02, 4.12499099e+03),
    dimnames=list(c("x", "y"), c("x", "y", "z")))

In [ ]:
# stash original coordinates
old <- spatialCoords(xen)
colData(xen)[c(".x", ".y")] <- old
# apply affine transformation
new <- old %*% t(mtx[, -3]) # scale/rotate &
new <- sweep(new, 2, mtx[, 3], `+`) # offset
spatialCoords(xen) <- new

In [ ]:
#| code-fold: true
df <- data.frame(spatialCoords(vis))
fd <- data.frame(spatialCoords(xen))
ggplot() + coord_equal() + theme_void() +
    geom_point(aes(x, y), df, col="grey", stroke=0, size=1) +
    geom_point(aes(x, y), fd, col="blue", stroke=0, size=0.1) 

## Aggregation

Binning single-cell resolution spatial data into spots can be useful for checking 
correlations between technical replicates of the same technology, identifying 
artifacts across technologies, and checking cell density (number of cells per spot). 
In general, it is possible to bin at the transcript-level (sub-cellular) or cell-level.

To aggregate single cell-level data from Xenium at the spot level, we first 
carry out a fixed-radius neighborhood search using `r BiocStyle::CRANpkg("RANN")` 
(see @sec-img-neighborhood-analysis) to identify, for every spot, cells whose centroid lies 
within a $\sim130$um distance (Visium spot diameter of 55um, divided by two 2, 
divided by 0.2115 = Xenium px size in um):

In [ ]:
# do a fixed-radius search to get cell 
# centroids that fall on a given spot
nns <- nn2(
    searchtype="radius", radius=55/2/0.2125, k=200,
    data=spatialCoords(xen), query=spatialCoords(vis))

Let's count and visualize the number of cells that overlap each spot:

In [ ]:
# get cell indices and number of cells per spot
vis$n_cells <- rowSums((idx <- nns$nn.idx) > 0)
plotCoords(vis, annotate="n_cells")

Next, we can aggregate single cell-level Xenium data into pseudo-spots. 
In addition, we propagate the Visium data's spatial coordinates, 
excluding spots without any overlapping cells:

In [ ]:
# aggregate Xenium data into pseudo-spots
ids <- rep.int(seq(ncol(vis)), vis$n_cells)
xem <- aggregateAcrossCells.se(xen[, c(t(idx))], ids)
xem <- as(as(xem, "SingleCellExperiment"), "SpatialExperiment")

# propagate Visium data's spatial coordinates, excluding empty pseudo-spots
spatialCoords(xem) <- spatialCoords(vis)[vis$n_cells > 0, ] 
colnames(xem) <- colnames(vis)[vis$n_cells > 0]
assays(xem) <- list(counts=assay(xem))
xem$in_tissue <- 1

Because we've aligned the Xenium to the Visium data, we can also 
propagate the Visium data's `imgData` (low resolution H&E staining) 
to the object containing pseudo-spot Xenium data:

In [ ]:
imgData(xem) <- imgData(vis)

Next, let's compute some standard quality control metrics on both, 
the Visium and pseudo-spot Xenium data. Besides dataset-specific metrics, 
we also specify the subset of genes that are shared between both datasets 
in order to obtain comparable metrics:

In [ ]:
sub <- list(gs=intersect(rownames(vis), rownames(xen)))
vis <- quickRnaQc.se(vis, subsets=sub)
xem <- quickRnaQc.se(xem, subsets=sub)

Let's visually compare the total counts per (pseudo-)spot between Visium and Xenium:

In [ ]:
# add counts of shared genes as observation metadata
vis$gs_sum <- vis$subset.proportion.gs * vis$sum
xem$gs_sum <- xem$subset.proportion.gs * xem$sum

# visualize side-by-side
plotVisium(vis, annotate="gs_sum", zoom=TRUE, facets=NULL) + ggtitle("Visium") +
plotVisium(xem, annotate="gs_sum", zoom=TRUE, facets=NULL) + ggtitle("Xenium")

We can also use a scatter plot - where points = (pseudo-)spots - 
to directly compare total counts between both technologies:

In [ ]:
df <- data.frame(
    n_cells=xem$counts,
    Xenium=xem$gs_sum,
    Visium=vis[, colnames(xem)]$gs_sum)

ggplot(df, aes(Xenium, Visium, col=n_cells)) + 
    scale_color_gradientn(colors=rev(hcl.colors(9, "Mako"))) +
    geom_point(alpha=0.5) + theme_bw() + theme(aspect.ratio=1) 

## Integration

Besides physically aligning both datasets, we can integrate them on 
a transcriptional level; here, using `r BiocStyle::Biocpkg("harmony")`.

To this end, we first consolidate the Visium and Xenium data into one object:

In [ ]:
# get shared features, observations & metadata
gs <- intersect(rownames(vis), rownames(xem))
cs <- intersect(colnames(vis), colnames(xem))
cd <- intersect(names(colData(vis)), names(colData(xem)))

# pool datasets together
lys <- list(Visium=vis, Xenium=xem)
lys <- mapply(spe=lys, sid=names(lys), \(spe, sid) {
    spe <- spe[gs, cs]
    spe$sample_id <- sid
    rowData(spe) <- NULL
    colData(spe) <- colData(spe)[cd]
    assay(spe) <- as(assay(spe), "dgCMatrix")
    return(spe)
}, SIMPLIFY=FALSE)
(obj <- do.call(cbind, lys))

In order to perform joint spatial clustering of both modalities, we construct
array coordinates for Xenium pseudo-spots that are offset from Visium spots:

In [ ]:
# offset the spatial location for joint clustering
# of Visium and adjacent pseudo-spot Xenium data
obj$array_row <- c(ar <- vis[, cs]$array_row, 100+ar)
obj$array_col <- c(ac <- vis[, cs]$array_col, 100+ac)

Next, we run a standard pipeline to perform log-library size normalization 
(using `r BiocStyle::Biocpkg("scrapper")`), principal component analysis (PCA), 
`harmony` integration, and dimension reduction (UMAP). Notably, the feature 
selection step that typically precedes PCA is skipped here, as the Xenium 
experiment includes only a curated selection of $\sim300$ targets by design.

In [ ]:
# minimal filtering
obj <- obj[, obj$gs_sum > 0]
# library size normalization 
obj <- normalizeRnaCounts.se(obj)
# principal component analysis
# (w/o additional feature selection)
obj <- runPca.se(obj, rownames(obj))
# 'harmony' integration
pcs <- RunHarmony(
    data_mat=reducedDim(obj, "PCA"), 
    meta_data=obj$sample_id, 
    verbose=FALSE)
reducedDim(obj, "PCA") <- pcs
# dimensionality reduction
map <- runUmap(t(pcs))
reducedDim(obj, "UMAP") <- map

In [ ]:
plotUMAP(obj, colour_by="sample_id", point_size=0.1) +
    guides(col=guide_legend(override.aes=list(alpha=1, size=2))) +
    theme_void() + theme(aspect.ratio=1, legend.key.size=unit(0, "pt"))

## Clustering

The `spatialCluster()` function clusters the spots, and adds the predicted cluster 
labels to the object. The authors recommend running with at least 10,000 iterations 
(`nrep=1e4`); we use fewer iterations in this demo for the sake of runtime. (Note 
that a random seed must be set (`set.seed()`) for the results to be reproducible.)

In [ ]:
# 'BayesSpace' clustering
res <- spatialCluster(obj, q=10, burn.in=100, nrep=1e3)
table(res$k <- factor(res$spatial.cluster))

We see high concordance between the two modalities, although the 
aggregated Xenium data appears to yield slightly higher granularity:

In [ ]:
pal <- hcl.colors(nlevels(res$k), "Spectral")
plotVisium(res, image=FALSE, annotate="k") +
    theme(legend.key.size=unit(0, "pt")) +
    scale_fill_manual(values=pal)

From here on out, both datasets could be analyzed together and/or independently, 
e.g., in order to identify cluster markers, annotate cell subpopulations etc.

## Appendix

### References {.unnumbered}